# Business Analytics Synthetic Data Analysis

This notebook explores a synthetic dataset containing two years of sales data.  

It is organised in increasing levels of complexity, starting with data loading and summarisation, followed by visual exploration and culminating in predictive modelling.  The goal is to demonstrate skills relevant for business analysts, program managers and data analysts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, mean_squared_error, confusion_matrix, accuracy_score, roc_auc_score, classification_report

# set aesthetics
sns.set(style='whitegrid', palette='muted', color_codes=True)

# Load dataset
df = pd.read_csv('data/synthetic_business_data.csv', parse_dates=['Date'])

# Display first few rows
df.head()

## Basic summary

Let's look at the shape of the data and basic descriptive statistics.

In [ ]:
print(f"Dataset contains {df.shape[0]} rows and {df.shape[1]} columns")

# Check for missing values
print('Missing values by column:')
print(df.isnull().sum())

# Summary statistics for numeric columns
df.describe(include='all')

## Exploratory data visualisation

We'll visualise distributions and relationships to understand sales patterns across regions, products and time.

In [ ]:
# Units sold by region
plt.figure(figsize=(6,4))
sns.barplot(x='Region', y='UnitsSold', data=df, estimator=np.mean)
plt.title('Average Units Sold by Region')
plt.ylabel('Average units sold')
plt.show()

In [ ]:
# Revenue by product
plt.figure(figsize=(6,4))
sns.barplot(x='Product', y='Revenue', data=df, estimator=np.mean)
plt.title('Average Revenue by Product')
plt.ylabel('Average revenue')
plt.show()

In [ ]:
# Revenue trend over time
df_monthly = df.resample('M', on='Date').sum(numeric_only=True)
plt.figure(figsize=(8,4))
plt.plot(df_monthly.index, df_monthly['Revenue'], marker='o')
plt.title('Monthly Revenue Over Time')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8,6))
num_cols = ['MarketingSpend','UnitsSold','Revenue','Satisfaction','Churn']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

## Predictive modelling – revenue

We'll build a linear regression model to predict revenue based on marketing spend, units sold, satisfaction and product/region categories.  

We use one‑hot encoding for categorical variables, split the data into training and test sets, fit the model and evaluate performance.

In [ ]:
# Prepare features and target for regression
X = df.copy()

# One-hot encode categorical variables
X = pd.get_dummies(X, columns=['Region','Product'], drop_first=True)

y = X['Revenue']
X = X.drop(['Revenue','Date','Churn'], axis=1)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train linear regression model
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Predict on test set
y_pred = lin_reg.predict(X_test)

# Evaluation metrics
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R-squared: {r2:.3f}")
print(f"RMSE: {rmse:.2f}")

# Display coefficients with feature names
coeffs = pd.Series(lin_reg.coef_, index=X.columns)
coeffs.sort_values(ascending=False)

## Predictive modelling – churn

Next, we'll build a logistic regression model to classify whether a customer will churn.  We use similar features as the regression model (excluding the churn column).  

We report accuracy, confusion matrix and AUC score to evaluate the model.

In [ ]:
# Prepare features and target for classification
X_clf = df.copy()

# One-hot encode categorical variables
X_clf = pd.get_dummies(X_clf, columns=['Region','Product'], drop_first=True)

y_clf = X_clf['Churn']
X_clf = X_clf.drop(['Churn','Date','Revenue'], axis=1)

# Split data
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

# Train logistic regression model
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(Xc_train, yc_train)

# Predict on test set
clf_pred = log_reg.predict(Xc_test)
clf_prob = log_reg.predict_proba(Xc_test)[:,1]

# Evaluation metrics
acc = accuracy_score(yc_test, clf_pred)
cm = confusion_matrix(yc_test, clf_pred)
roc_auc = roc_auc_score(yc_test, clf_prob)

print(f"Accuracy: {acc:.3f}")
print(f"ROC AUC: {roc_auc:.3f}")
print("Confusion matrix:\n", cm)

# Classification report
print(classification_report(yc_test, clf_pred))